In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("fertilizer.csv")
df.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26,52,38,Sandy,Maize,37,0,0,Urea
1,29,52,45,Loamy,Sugarcane,12,0,36,DAP
2,34,65,62,Black,Cotton,7,9,30,14-35-14
3,32,62,34,Red,Tobacco,22,0,20,28-28
4,28,54,46,Clayey,Paddy,35,0,0,Urea


In [3]:
df.columns

Index(['Temparature', 'Humidity ', 'Moisture', 'Soil Type', 'Crop Type',
       'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer Name'],
      dtype='object')

In [4]:
soil = LabelEncoder()
crop = LabelEncoder()
target = LabelEncoder()

df["Soil Type"] = soil.fit_transform(df["Soil Type"])
df["Crop Type"] = crop.fit_transform(df["Crop Type"])
df["Fertilizer Name"] = target.fit_transform(df["Fertilizer Name"])

In [5]:
X = df.drop("Fertilizer Name", axis=1)
y = df["Fertilizer Name"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1
)

model.fit(X_train, y_train)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [8]:
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)

print("Accuracy:", acc)

Accuracy: 1.0


In [9]:
joblib.dump(model, "fertilizer_model.pkl")
joblib.dump(soil, "soil_encoder.pkl")
joblib.dump(crop, "crop_encoder.pkl")
joblib.dump(target, "target_encoder.pkl")

['target_encoder.pkl']

In [10]:
sample = pd.DataFrame([{
    "Temparature": 26,
    "Humidity ": 52,
    "Moisture": 38,
    "Soil Type": soil.transform(["Loamy"])[0],
    "Crop Type": crop.transform(["Wheat"])[0],
    "Nitrogen": 37,
    "Potassium": 0,
    "Phosphorous": 0
}])

pred = model.predict(sample)
target.inverse_transform(pred)

array(['Urea'], dtype=object)

In [11]:
def fertilizer_amount(crop, area, fertilizer):
    
    chart = {
        "Urea": 45,
        "DAP": 50,
        "14-35-14": 40,
        "28-28": 35
    }

    base = chart.get(fertilizer, 30)

    return base * area

In [12]:
fert = "Urea"
area = 2

amt = fertilizer_amount("Wheat", area, fert)

print("Recommended:", amt, "kg")

Recommended: 90 kg


In [13]:
if amt > 100:
    print("Warning: Excess fertilizer may damage soil")

In [14]:
fert = target.inverse_transform(pred)[0]

area = 2

amt = fertilizer_amount("Wheat", area, fert)

print("Fertilizer:", fert)
print("Amount:", amt, "kg")

Fertilizer: Urea
Amount: 90 kg


In [15]:
def get_quantity(row):
    
    if row["Fertilizer Name"] == "Urea":
        return row["Nitrogen"] * 1.8
        
    elif row["Fertilizer Name"] == "DAP":
        return row["Phosphorous"] * 1.5
        
    elif row["Fertilizer Name"] == "Potash":
        return row["Potassium"] * 1.7
        
    else:
        return (row["Nitrogen"] + row["Phosphorous"] + row["Potassium"]) * 1.2

df["Quantity"] = df.apply(get_quantity, axis=1)

In [16]:
df[["Fertilizer Name","Quantity"]].head()

,Fertilizer Name,Quantity
0,6,44.4
1,5,57.6
2,1,55.2
3,4,50.4
4,6,42.0


In [17]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

In [18]:
X = df.drop(["Fertilizer Name","Quantity"], axis=1)
y = df["Quantity"]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
q_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1
)

q_model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [21]:
pred = q_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)

print("MAE:", mae)

MAE: 2.071426181793213


In [22]:
joblib.dump(q_model, "quantity_model.pkl")

['quantity_model.pkl']

In [23]:
sample = pd.DataFrame([{
    "Temparature": 26,
    "Humidity ": 52,
    "Moisture": 38,
    "Soil Type": soil.transform(["Loamy"])[0],
    "Crop Type": crop.transform(["Wheat"])[0],
    "Nitrogen": 37,
    "Potassium": 0,
    "Phosphorous": 0
}])

qty = q_model.predict(sample)

print("Recommended Quantity:", round(qty[0],2), "kg/acre")

Recommended Quantity: 45.62 kg/acre


In [24]:
if qty[0] > 80:
    print("Warning: High fertilizer use")

In [25]:
area = 3   # acre

total_qty = qty[0] * area

print("Per Acre:", round(qty[0],2), "kg")
print("Total Needed:", round(total_qty,2), "kg")

Per Acre: 45.62 kg
Total Needed: 136.85 kg


In [26]:
if sample["Nitrogen"][0] < 20:
    print("Nitrogen Deficiency")

if sample["Phosphorous"][0] < 20:
    print("Phosphorous Deficiency")

if sample["Potassium"][0] < 20:
    print("Potassium Deficiency")

Phosphorous Deficiency
Potassium Deficiency


In [27]:
fert_name = target.inverse_transform(model.predict(sample))[0]

print("Recommended Fertilizer:", fert_name)
print("Recommended Quantity:", round(qty[0],2), "kg/acre")
print("Total for 3 Acre:", round(total_qty,2), "kg")

Recommended Fertilizer: Urea
Recommended Quantity: 45.62 kg/acre
Total for 3 Acre: 136.85 kg
